In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile
import shutil
import os

drive.mount("/content/drive", force_remount=False)

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No archive uploaded.")

zip_path = Path("/content") / list(uploaded.keys())[0]

WORK_ROOT = Path("/content/roadwatch")
EXTRACT_ROOT = Path("/content/_roadwatch_extract")

shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
shutil.rmtree(WORK_ROOT, ignore_errors=True)

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    for member in z.infolist():
        fixed_name = member.filename.replace("\\", "/")
        target_path = EXTRACT_ROOT / fixed_name
        if fixed_name.endswith("/"):
            target_path.mkdir(parents=True, exist_ok=True)
        else:
            target_path.parent.mkdir(parents=True, exist_ok=True)
            with z.open(member) as src, open(target_path, "wb") as dst:
                dst.write(src.read())

required_files = {"train.py", "prepare_subset.py", "evaluate.py", "export_onnx.py"}
candidate_dirs = []

for folder in EXTRACT_ROOT.rglob("*"):
    if folder.is_dir():
        existing_files = {p.name for p in folder.iterdir() if p.is_file()}
        if required_files.issubset(existing_files):
            candidate_dirs.append(folder)

if not candidate_dirs:
    raise RuntimeError("Required model scripts were not found in the archive.")

SRC_DIR = candidate_dirs[0]

for item in SRC_DIR.iterdir():
    destination = WORK_ROOT / item.name
    if item.is_dir():
        shutil.copytree(item, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(item, destination)

(WORK_ROOT / "data").mkdir(exist_ok=True)

print("Workspace:", WORK_ROOT)
!find /content/roadwatch -maxdepth 2 -type f -print


In [ ]:
%cd /content/roadwatch

import os
import torch
import ultralytics
from ultralytics import settings

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

!pip -q install ultralytics kaggle pyyaml pandas matplotlib opencv-python onnx
!pip -q uninstall -y wandb

try:
    settings.update({"wandb": False})
except Exception:
    pass

print("PyTorch version:", torch.__version__)
print("Ultralytics version:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU runtime is required.")


/content/roadwatch
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 126.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch version: 2.11.0+cu128
Ultralytics version: 8.4.118
CUDA available: True
GPU name: NVIDIA A100-SXM4-80GB


In [ ]:
from getpass import getpass
from pathlib import Path
import os

kaggle_token = getpass("Kaggle API token: ").strip()
if not kaggle_token:
    raise RuntimeError("Kaggle token is empty.")

kaggle_dir = Path("/root/.kaggle")
kaggle_dir.mkdir(parents=True, exist_ok=True)

token_path = kaggle_dir / "access_token"
token_path.write_text(kaggle_token)
os.chmod(token_path, 0o600)

os.environ["KAGGLE_API_TOKEN"] = kaggle_token

print("Kaggle credentials configured.")


In [ ]:
from pathlib import Path
import os

%cd /content/roadwatch

raw_root = Path("/content/rdd2022_raw")
train_images = raw_root / "train/images"
val_images = raw_root / "val/images"

if not train_images.exists() or not val_images.exists():
    raw_root.mkdir(parents=True, exist_ok=True)
    token_path = Path("/root/.kaggle/access_token")
    if token_path.exists():
        os.environ["KAGGLE_API_TOKEN"] = token_path.read_text().strip()
    !kaggle datasets download -d sreekaraditya/rdd2022-yolo-crackscan-v2 -p /content/rdd2022_raw --unzip

print("Train images:", train_images, train_images.exists())
print("Val images:", val_images, val_images.exists())

!find /content/rdd2022_raw/train/images -type f | wc -l
!find /content/rdd2022_raw/val/images -type f | wc -l


/content/roadwatch
Dataset URL: https://www.kaggle.com/datasets/sreekaraditya/rdd2022-yolo-crackscan-v2
License(s): CC-BY-SA-4.0
100% 12.3G/12.3G [05:15<00:00, 42.0MB/s]


Dataset paths:
Train images: /content/rdd2022_raw/train/images EXISTS
Val images: /content/rdd2022_raw/val/images EXISTS

Image counts:
32628
5757


In [ ]:
%cd /content/roadwatch

from pathlib import Path
import yaml

train_images = Path("/content/rdd2022_raw/train/images")
val_images = Path("/content/rdd2022_raw/val/images")

if not train_images.exists() or not val_images.exists():
    raise RuntimeError("Dataset paths are missing.")

!rm -rf /content/roadwatch/data/rdd2022_large

!python prepare_subset.py \
  --train_images /content/rdd2022_raw/train/images \
  --val_images /content/rdd2022_raw/val/images \
  --out_dir data/rdd2022_large \
  --max_train 15000 \
  --max_val 3000

print("Train subset:")
!find /content/roadwatch/data/rdd2022_large/images/train -type f | wc -l

print("Validation subset:")
!find /content/roadwatch/data/rdd2022_large/images/val -type f | wc -l

!cat /content/roadwatch/data/rdd2022_large/data.yaml

with open("/content/roadwatch/data/rdd2022_large/data.yaml", "r") as f:
    data = yaml.safe_load(f)

expected_classes = ["Longitudinal", "Transverse", "Alligator", "Pothole"]

if data.get("names") != expected_classes:
    raise RuntimeError("Unexpected class order: %s" % data.get("names"))

print("Class order verified.")


/content/roadwatch
[train] copied 15000 pairs
[val] copied 3000 pairs
data.yaml written at /content/roadwatch/data/rdd2022_large/data.yaml

Subset counts:
Train images:
15000
Val images:
3000

Subset data.yaml:
path: /content/roadwatch/data/rdd2022_large
train: images/train
val: images/val
names:
- Longitudinal
- Transverse
- Alligator
- Pothole
Class order verified.


In [ ]:
%cd /content/roadwatch

import os
import csv
from pathlib import Path
from ultralytics import YOLO, settings

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

try:
    settings.update({"wandb": False})
except Exception:
    pass

DRIVE_ROOT = Path("/content/drive/MyDrive/RoadWatch_Model_Training")
RUN_PROJECT = DRIVE_ROOT / "runs/roadwatch"
RUN_NAME = "yolov8s_rdd2022_60ep"
RUN_DIR = RUN_PROJECT / RUN_NAME
WEIGHTS_DIR = RUN_DIR / "weights"
BEST_PT = WEIGHTS_DIR / "best.pt"
LAST_PT = WEIGHTS_DIR / "last.pt"
DATA_YAML = Path("/content/roadwatch/data/rdd2022_large/data.yaml")
TARGET_EPOCHS = 60

RUN_PROJECT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "logs").mkdir(parents=True, exist_ok=True)

results_csv = RUN_DIR / "results.csv"
completed_epochs = 0

if results_csv.exists():
    with open(results_csv, "r") as f:
        rows = list(csv.DictReader(f))
    completed_epochs = len(rows)

remaining_epochs = max(TARGET_EPOCHS - completed_epochs, 0)

print("Run directory:", RUN_DIR)
print("Completed epochs:", completed_epochs)
print("Remaining epochs:", remaining_epochs)
print("best.pt exists:", BEST_PT.exists())
print("last.pt exists:", LAST_PT.exists())

if completed_epochs >= TARGET_EPOCHS and BEST_PT.exists():
    print("Training checkpoint is complete.")
elif LAST_PT.exists():
    try:
        model = YOLO(str(LAST_PT))
        model.train(resume=True)
    except Exception as exc:
        remaining_epochs = max(TARGET_EPOCHS - completed_epochs, 1)
        continued_name = "yolov8s_rdd2022_60ep_continued_from_epoch_%d" % completed_epochs
        print("Resume fallback:", repr(exc))
        model = YOLO(str(LAST_PT))
        model.train(
            data=str(DATA_YAML),
            epochs=remaining_epochs,
            imgsz=640,
            batch=-1,
            workers=4,
            patience=15,
            project=str(RUN_PROJECT),
            name=continued_name,
            exist_ok=True,
        )
else:
    model = YOLO("yolov8s.pt")
    model.train(
        data=str(DATA_YAML),
        epochs=TARGET_EPOCHS,
        imgsz=640,
        batch=-1,
        workers=4,
        patience=15,
        project=str(RUN_PROJECT),
        name=RUN_NAME,
        exist_ok=True,
    )

print("Training step completed.")


/content/roadwatch
Saved run dir: /content/drive/MyDrive/RoadWatch_Model_Training/runs/roadwatch/yolov8s_rdd2022_60ep
Completed epochs found in Drive: 26
Existing best.pt: True
Existing last.pt: True

Training resumed from saved last.pt checkpoint.

EarlyStopping: Training stopped early as no improvement was observed in the last 15 epochs.
Best results were observed at epoch 44 and saved as best.pt.

33 epochs completed in 0.881 hours.

Validation summary for best.pt:
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)
                   all       3000       4367      0.582      0.513      0.532      0.258
          Longitudinal       1042       2098      0.592      0.524      0.543      0.284
            Transverse        623        943      0.573      0.534      0.536      0.249
             Alligator        634        793      0.613      0.614      0.638      0.328
               Pothole        301        533      0.550      0.379      0.413     

In [ ]:
%cd /content/roadwatch

import os
import shutil
import zipfile
import subprocess
from pathlib import Path
from google.colab import files

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

DRIVE_ROOT = Path("/content/drive/MyDrive/RoadWatch_Model_Training")
RUN_PROJECT = DRIVE_ROOT / "runs/roadwatch"
FINAL_DIR = DRIVE_ROOT / "final_artifacts"
LOG_DIR = DRIVE_ROOT / "logs"
ZIP_PATH = DRIVE_ROOT / "roadwatch_yolov8s_final_artifacts.zip"
DATA_YAML = Path("/content/roadwatch/data/rdd2022_large/data.yaml")

FINAL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

best_candidates = sorted(
    RUN_PROJECT.rglob("weights/best.pt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not best_candidates:
    raise RuntimeError("No best.pt checkpoint found.")

BEST_PT = best_candidates[0]
RUN_DIR = BEST_PT.parents[1]
WEIGHTS_DIR = BEST_PT.parent
LAST_PT = WEIGHTS_DIR / "last.pt"

print("Selected checkpoint:", BEST_PT)
print("Selected run:", RUN_DIR)

eval_cmd = "cd /content/roadwatch && python evaluate.py --weights '%s' --data '%s' --imgsz 640 2>&1 | tee '%s'" % (
    BEST_PT,
    DATA_YAML,
    LOG_DIR / "final_evaluation.log",
)
eval_result = subprocess.run(eval_cmd, shell=True)

if eval_result.returncode != 0:
    print("Evaluation command returned non-zero status.")

export_cmd = "cd /content/roadwatch && python export_onnx.py --weights '%s' --imgsz 640 --runs 50 2>&1 | tee '%s'" % (
    BEST_PT,
    LOG_DIR / "final_onnx_export_latency.log",
)
export_result = subprocess.run(export_cmd, shell=True)

if export_result.returncode != 0:
    print("ONNX export command returned non-zero status.")

for item in FINAL_DIR.iterdir():
    if item.is_file():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

def copy_if_exists(src, dst_dir=FINAL_DIR):
    src = Path(src)
    if src.exists():
        dst = dst_dir / src.name
        shutil.copy2(src, dst)
        print("Copied:", dst)
    else:
        print("Missing:", src)

copy_if_exists(BEST_PT)
copy_if_exists(LAST_PT)

for filename in [
    "results.csv",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "P_curve.png",
    "R_curve.png",
    "F1_curve.png",
    "labels.jpg",
    "args.yaml",
]:
    copy_if_exists(RUN_DIR / filename)

for onnx_file in WEIGHTS_DIR.glob("*.onnx"):
    copy_if_exists(onnx_file)

copy_if_exists(LOG_DIR / "final_evaluation.log")
copy_if_exists(LOG_DIR / "final_onnx_export_latency.log")

summary = "\n".join([
    "RoadWatch YOLOv8s model artifacts",
    "",
    "Final model:",
    str(FINAL_DIR / "best.pt"),
    "",
    "Run directory:",
    str(RUN_DIR),
    "",
    "Source checkpoint:",
    str(BEST_PT),
    "",
    "Class order:",
    "0 = Longitudinal",
    "1 = Transverse",
    "2 = Alligator",
    "3 = Pothole",
])

(FINAL_DIR / "README_FINAL_MODEL.txt").write_text(summary)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for path in FINAL_DIR.rglob("*"):
        if path.is_file():
            z.write(path, arcname="final_artifacts/%s" % path.name)
    for path in RUN_DIR.rglob("*"):
        if path.is_file():
            z.write(path, arcname="run/%s" % path.relative_to(RUN_DIR))

print("ZIP:", ZIP_PATH)
print("ZIP size MB:", round(ZIP_PATH.stat().st_size / (1024 * 1024), 2))

try:
    files.download(str(ZIP_PATH))
except Exception as exc:
    print("Download skipped:", repr(exc))

print("Artifact collection completed.")


/content/roadwatch
Selected best.pt: /content/drive/MyDrive/RoadWatch_Model_Training/runs/roadwatch/yolov8s_rdd2022_60ep/weights/best.pt
Selected run dir: /content/drive/MyDrive/RoadWatch_Model_Training/runs/roadwatch/yolov8s_rdd2022_60ep

Running evaluation...

Exporting ONNX and benchmarking latency...

Copied: /content/drive/MyDrive/RoadWatch_Model_Training/final_artifacts/best.pt
Copied: /content/drive/MyDrive/RoadWatch_Model_Training/final_artifacts/last.pt
Copied: /content/drive/MyDrive/RoadWatch_Model_Training/final_artifacts/results.csv
Copied: /content/drive/MyDrive/RoadWatch_Model_Training/final_artifacts/results.png
Copied: /content/drive/MyDrive/RoadWatch_Model_Training/final_artifacts/confusion_matrix.png
Copied: /content/drive/MyDrive/RoadWatch_Model_Training/final_artifacts/confusion_matrix_normalized.png
Copied: /content/drive/MyDrive/RoadWatch_Model_Training/final_artifacts/labels.jpg
Copied: /content/drive/MyDrive/RoadWatch_Model_Training/final_artifacts/args.yaml
Cop

In [ ]:
from pathlib import Path
import csv

DRIVE_ROOT = Path("/content/drive/MyDrive/RoadWatch_Model_Training")
FINAL_DIR = DRIVE_ROOT / "final_artifacts"
ZIP_PATH = DRIVE_ROOT / "roadwatch_yolov8s_final_artifacts.zip"
RUN_PROJECT = DRIVE_ROOT / "runs/roadwatch"

print("Artifacts folder:", FINAL_DIR, FINAL_DIR.exists())
print("Final model:", FINAL_DIR / "best.pt", (FINAL_DIR / "best.pt").exists())

if (FINAL_DIR / "best.pt").exists():
    print("Model size MB:", round((FINAL_DIR / "best.pt").stat().st_size / (1024 * 1024), 2))

print("ZIP file:", ZIP_PATH, ZIP_PATH.exists())

if ZIP_PATH.exists():
    print("ZIP size MB:", round(ZIP_PATH.stat().st_size / (1024 * 1024), 2))

print("Training runs:")
for results_csv in sorted(RUN_PROJECT.rglob("results.csv")):
    try:
        with open(results_csv, "r") as f:
            rows = list(csv.DictReader(f))
        print(results_csv)
        print("epochs:", len(rows))
        if rows:
            print("last row:", rows[-1])
    except Exception as exc:
        print(results_csv, repr(exc))

print("Final checkpoint path:")
print(FINAL_DIR / "best.pt")


Final artifacts folder: /content/drive/MyDrive/RoadWatch_Model_Training/final_artifacts EXISTS
Final best.pt: /content/drive/MyDrive/RoadWatch_Model_Training/final_artifacts/best.pt EXISTS
best.pt size MB: 21.48
Final ZIP: /content/drive/MyDrive/RoadWatch_Model_Training/roadwatch_yolov8s_final_artifacts.zip EXISTS
ZIP size MB: 157.81

Available training runs:
- /content/drive/MyDrive/RoadWatch_Model_Training/runs/roadwatch/yolov8s_rdd2022_60ep/results.csv
  epoch rows: 59
  last row: {'epoch': '59', 'time': '3172.77', 'train/box_loss': '1.30209', 'train/cls_loss': '0.98788', 'train/dfl_loss': '1.27982', 'metrics/precision(B)': '0.61232', 'metrics/recall(B)': '0.50925', 'metrics/mAP50(B)': '0.52022', 'metrics/mAP50-95(B)': '0.25255', 'val/box_loss': '1.85109', 'val/cls_loss': '1.65153', 'val/dfl_loss': '1.62321', 'lr/pg0': '0.00129', 'lr/pg1': '0.00043', 'lr/pg2': '0.00129', 'lr/pg3': '0.00043', 'lr/pg4': '0.00129', 'lr/pg5': '0.00043', 'lr/pg6': '0.00129', 'lr/pg7': '0.00043'}
